# GB-Agent 项目复盘


## 项目解决什么问题

GB-Agent 是一个面向标准领域的本地智能代理平台，解决标准文档数量多、查询困难、审查流程重复、标准引用关系复杂等问题。用户可以上传 PDF、DOCX 或 TXT 标准文件，建立本地标准知识库，然后完成语义检索、标准问答、条款审查、合规分析、内容生成、监控订阅和批量处理。

项目主要用户包括标准起草者、审查者、合规人员、产品设计人员、认证机构和企业用户。不同角色共享同一套数据与工具能力，但快捷操作和回复风格不同。

项目的核心价值不是单纯调用大模型，而是把文档处理、检索、领域工具、任务编排、结果溯源和前端交互组合成完整工作流，尽可能包括所有与标准相关的工作。

### 一分钟项目介绍

> GB-Agent是一个面向标准领域的本地智能代理平台。我负责整体架构和全栈开发。后端使用FastAPI，前端使用原生JavaScript；文档经过解析、清洗和切块后，由BGE生成向量并存入ChromaDB，SQLite保存业务数据。Agent层使用LangGraph，根据用户意图生成执行计划并调用检索、审查和生成工具，最后通过SSE把执行进度、回答及来源推送给前端。

**面试提示**：先说业务问题和个人职责，再说架构与技术。不要一上来罗列框架名称。

## 系统总体架构

```mermaid
flowchart TB
    U[用户浏览器] --> F[Vanilla JavaScript SPA]
    F -->|HTTP / SSE| A[FastAPI 路由层]
    A --> O[LangGraph Agent 编排器]
    O --> T[领域工具注册表]
    T --> R[检索与文档服务]
    T --> X[审查、分析、生成、监控工具]
    R --> E[BGE Embedding]
    R --> C[ChromaDB 向量库]
    R --> S[SQLite + FTS5]
    O --> L[llama.cpp 本地 LLM]
    A --> M[会话、记忆、日志、配置]
```

| 层级 | 技术 | 主要职责 |
| --- | --- | --- |
| 前端 | HTML、CSS、Vanilla JavaScript | 单页交互、上传、对话、计划与来源展示 |
| API | FastAPI、Pydantic、Uvicorn | 路由、参数校验、异常处理、SSE 响应 |
| Agent | LangGraph、LangChain 工具接口 | 状态管理、意图识别、规划、工具执行和合成 |
| 文档 | PyMuPDF、python-docx、Marker OCR | 解析、清洗、元数据提取和切块 |
| 检索 | BGE、ChromaDB、jieba、FTS5 | 语义检索与精确关键词检索 |
| 数据 | SQLite、ChromaDB | 业务数据、会话、记忆、原文块和向量 |
| 模型 | llama.cpp、GGUF、Qwen 2.5 | 本地回答合成，支持断网部署 |
| 工程 | uv、pytest、日志、配置文件 | 依赖管理、测试、运行和排障 |

主要代码入口：

- `backend/main.py`：FastAPI 应用入口。
- `backend/routes/`：HTTP API 路由。
- `agent/orchestrator.py`：LangGraph 主编排图。
- `agent/tools/`：领域工具。
- `document/`：文档清洗、切块和入库。
- `backend/database/`：SQLite 与 ChromaDB 操作。
- `static/js/`：前端业务模块。

## 一次 Agent 对话如何执行

用户在前端发送问题后，`static/js/chat.js` 向 `POST /api/agent/chat` 发送 JSON 请求。`backend/routes/agent.py` 创建或读取会话，构造初始 AgentState，然后调用 `build_agent_graph()` 创建 LangGraph。

```mermaid
flowchart LR
    A[用户问题] --> B[intent 意图识别]
    B --> C[planner 生成计划]
    C -->|简单任务| D[tool_executor]
    D -->|还有步骤| D
    D --> E[synthesize]
    C -->|复杂任务| F[dag_executor]
    F --> E
    C -->|普通对话| G[reply]
    C -->|置信度过低| H[ask]
    E --> I[最终回答]
```

### 核心节点

| 节点 | 作用 |
| --- | --- |
| `intent` | 识别意图、用户角色和任务复杂度，同时召回长期记忆 |
| `planner` | 根据意图生成展示计划、工具执行计划或 DAG |
| `tool_executor` | 顺序调用工具，累计结果、来源、耗时和执行状态 |
| `dag_executor` | 按节点依赖执行复杂任务，支持并行条件、重试和回退 |
| `synthesize` | 汇总工具结果，选择本地 LLM 合成或规则摘要 |
| `reply` | 不需要工具时返回角色化普通回复 |
| `ask` | 意图置信度过低时要求用户补充信息 |

### AgentState 保存什么

AgentState 是节点之间共享的状态，主要包含：用户问题、意图、角色、复杂度、计划、当前步骤、工具参数、工具结果、来源、会话 ID、记忆、最终回答、建议和最大迭代次数。

LangGraph 的价值在于把长任务拆成显式节点，并统一管理状态、路由、循环和结束条件。普通函数也能实现相同业务，但随着节点、条件和恢复逻辑增加，代码会更难维护。

**面试提示**：如果被问“为什么使用 LangGraph”，回答状态管理、显式流程、可观测性和扩展性，不要只说它是热门框架。

## 规划、Tool Calling 与工具系统

项目使用统一工具抽象 `GBAgentTool`。每个工具声明名称、描述、参数和所属领域，并实现异步 `execute()`。`ToolRegistry` 负责注册、查询和列出工具，编排器按工具名称找到实例并执行。

工具覆盖知识检索、分析诊断、内容生成、校验核查、监控、管理、协作和认证等领域。当前代码中的工具类数量已经超过 README 中早期记录的 22 个，因此介绍时最好说“数十个领域工具”，或者以当前代码重新统计，不要死记旧数字。

### 当前规划机制的真实边界

当前主链路的规划主要由规则和预定义模板驱动：

1. 意图分类得到 `query`、`review`、`draft`、`compare` 等类型。
2. `_build_execution_plan()` 根据意图选择预设工具和参数。
3. 复杂任务由 `build_dag_plan()` 生成预定义 DAG。
4. 编排器按照计划调用工具。

虽然注册表支持把工具转换成 LangChain StructuredTool，但当前主调用链并不是由 LLM 读取所有工具描述后自主决定每一次调用。

正确表述：

> 基于 LangGraph 实现 Plan-and-Execute 工作流，根据意图和任务复杂度生成线性计划或 DAG 计划，通过统一工具注册表执行领域工具，并加入超时、重试、回退和结果汇总机制。

不应表述为：

> 大模型可以从全部工具中完全自主选择工具并动态生成任意计划。

### 为什么使用规则规划

标准审查属于专业和高风险场景。规则规划的可解释性、确定性和可测试性更强，能够保证指定检查一定执行。代价是灵活性不足，新意图和流程需要修改代码。后续可以使用 LLM 结构化输出生成计划，再通过工具白名单、参数模型、最大步数和人工确认控制风险。

## 文档如何进入知识库

```mermaid
flowchart LR
    A[上传文件] --> B[格式与大小校验]
    B --> C[保存文件并创建文档记录]
    C --> D[解析为文本或 Markdown]
    D --> E[清洗文本]
    E --> F[提取标准元数据]
    F --> G[按标题层级切块]
    G --> H[BGE 批量生成向量]
    H --> I[SQLite 保存原文与元数据]
    H --> J[ChromaDB 保存向量]
    I --> K[FTS5 建立关键词索引]
```

### 文档解析

- PDF：优先使用 PyMuPDF，速度快且依赖较轻。
- 扫描 PDF：平均每页文字过少时回退 Marker OCR。
- DOCX：使用 python-docx 提取段落和表格。
- TXT：依次尝试 UTF-8、GBK、GB2312、GB18030 等编码。

### 切块策略

系统根据标准文档的 Markdown 标题结构切块：

| 标题 | 含义 | 处理方式 |
| --- | --- | --- |
| `#` | 标准名称 | 保存为 `standard_name` |
| `##` | 章标题 | 保存为父级上下文 |
| `###` | 节标题 | 作为主要块边界 |

目标块大小约为 100～600 tokens。过长内容继续按段落拆分；没有标题结构时，使用段落与约 500 tokens 上限进行兜底切块。

按语义标题切块比固定字符切块更适合标准文档，因为标题、章节和条款关系会被保留。但当前 token 数是经验公式估算，不是真实 tokenizer 计数，这是后续可优化点。

**面试提示**：面试官问切块大小时，不要只报数字。要说明块太小会缺失上下文，太大会引入噪声、增加上下文成本，并降低检索粒度。

## RAG 与检索链路

RAG 的目的不是让模型记住上传文档，而是在回答问题前检索相关原文，把原文作为上下文交给模型，从而减少知识缺失和幻觉。

当前语义问答链路：

```text
用户问题
  → BGE 生成归一化查询向量
  → ChromaDB 查询 Top-K 文本块
  → 拼接文本上下文和来源
  → 工具返回结构化结果
  → synthesize 节点调用本地 LLM 生成回答
  → 前端展示回答与来源
```

### 三种数据能力

| 组件 | 保存什么 | 用途 |
| --- | --- | --- |
| SQLite `vector_chunks` | 原文、标题、标准号、token 数 | 业务查询、关联和原文持久化 |
| ChromaDB | 向量、原文副本、元数据 | 语义相似度检索 |
| SQLite FTS5 | jieba 分词后的索引 | 标准号、引用链等关键词精确检索 |

### 当前是否属于混合检索

主搜索接口、`search_standards` 和 `qa_with_context` 当前都以 ChromaDB 向量检索为主。FTS5 已经建立并用于部分引用关系场景，但还没有把向量结果和关键词结果统一召回、融合打分和重排。

因此正确表述是：

> 系统以 ChromaDB 语义检索作为 RAG 主链路，同时构建了基于 jieba 和 SQLite FTS5 的关键词索引，用于标准编号和引用关系等精确检索场景；统一混合召回与融合排序是后续优化方向。

### 当前缺少的高级 RAG 能力

- 查询改写和多路召回。
- 向量与关键词结果融合。
- Cross-Encoder Reranker 重排。
- 文本块去重和上下文压缩。
- 根据问题动态选择 Top-K。
- 完整评测集、Recall@K、MRR 和回答忠实度指标。
- 回答句子与原始条款的严格引用对齐。

## SQLite、ChromaDB 与数据一致性

SQLite 保存业务实体，包括文档、文本块、用户、会话、记忆、订阅、告警、知识图谱节点和批处理任务。ChromaDB 专门保存向量并执行相似度检索。

项目没有把所有数据都放进 ChromaDB，原因是向量数据库擅长相似度搜索，却不适合承担复杂事务、外键关系、状态流转和普通条件查询。SQLite 简单、无需独立服务，适合本地单机部署。

### SQLite 工程设置

- 开启 WAL，提高读写并发能力。
- 开启外键约束，维护关联完整性。
- 设置 `busy_timeout`，减少短暂写锁导致的失败。
- 按线程维护连接，避免在线程池任务之间错误共享连接状态。
- 使用幂等迁移补充表和字段。

### 双写风险

入库时需要同时写 SQLite、ChromaDB 和 FTS5。它们之间没有分布式事务，因此中途失败可能造成部分写入。当前代码通过状态字段、异常捕获和删除逻辑降低风险，但仍需要考虑补偿事务、幂等重试和定期一致性检查。

**面试提示**：如果被问“为什么不直接使用 PostgreSQL”，可以回答本地单用户部署优先考虑零运维和文件级持久化；如果转为多人服务器版本，应考虑 PostgreSQL、连接池、并发事务、权限隔离和备份恢复。

## FastAPI、异步任务与 SSE

FastAPI 负责路由、请求模型校验、异常转换和响应。应用启动时初始化数据库、可选地预热嵌入模型，并挂载静态前端；退出时关闭数据库连接。

文档解析、Embedding 和 llama.cpp 推理包含同步且耗时的工作。如果直接在异步路由中调用，会阻塞事件循环。项目使用 `asyncio.to_thread()` 或线程执行器把同步任务移出事件循环，并使用超时限制防止任务无限等待。

### 为什么使用 SSE

Agent 请求可能需要经过意图识别、多个工具和模型合成，用户不能长时间只看加载动画。SSE 允许服务器通过一个 HTTP 连接持续向浏览器推送事件。

项目推送的主要事件包括：

- `agent_plan`：执行计划。
- `intent_result`：意图识别结果。
- `tool_call_start`、`tool_call_progress`、`tool_result`：工具状态。
- `plan_node_update`、`plan_node_result`：DAG 节点状态。
- `text`：回答内容。
- `sources`：来源。
- `suggestion`：后续操作建议。
- `error`：错误信息。

SSE 适合服务器到浏览器的单向事件流，实现简单并能自动沿用 HTTP 基础设施。WebSocket 适合真正的双向实时通信。项目中的回答文本并非底层模型原生 token 流，而是模型完成生成后按字符块模拟流式输出；面试时要如实说明。

## 前端如何处理流式响应

前端使用模块化原生 JavaScript，没有构建工具。`chat.js` 使用 `fetch()` 发起请求，通过 `ReadableStream.getReader()` 读取响应字节，再使用 `TextDecoder` 解码。

SSE 数据可能在任意字节位置被分割，所以前端必须维护 `buffer`：每次把新文本追加到缓冲区，按换行拆分，只处理完整行，把最后一个不完整片段留到下一次读取。

不同事件被分派给不同 UI 更新函数：计划事件更新步骤列表，工具事件更新进度，文本事件更新回答气泡，来源事件更新上下文面板。`AbortController` 用于停止生成。

### 原生 JavaScript 方案的取舍

优点：部署简单、零构建、依赖少，适合本地工具。缺点：随着页面状态和模块增多，手动 DOM 操作、共享状态和生命周期管理会变复杂。后续可以用 React 重构对话、上传和工作台组件，但重构前应先明确状态边界和 API 契约。

**面试提示**：招聘要求前端框架时，不要把 Vanilla JavaScript 说成 React 经验。可以强调自己理解浏览器 API、异步请求和流式解析，并说明正在用 React 重构现有前端。

## 本地模型、配置与涉密部署

项目使用 Pydantic Settings 从 `.env` 读取配置，包括模型路径、上下文长度、线程数、温度、检索 Top-K、数据库路径、上传限制、日志等级和服务端口。配置集中管理比在代码中散落常量更容易部署和测试。

LLM 客户端通过 llama-cpp-python 加载 GGUF 模型，并使用异步锁保证只加载一次，使用信号量限制并发推理，避免内存不足。模型加载和推理被放到线程执行器，外层设置超时。

`DEPLOY_MODE=airgap` 会进入断网模式，外部网络访问被中间件限制，嵌入模型和 LLM 都必须从本地文件加载。这适合标准、政企或涉密文档场景。

需要了解的部署与排障问题：

- 模型文件是否存在，Embedding 维度是否与索引一致。
- 服务端口是否被占用。
- 上传目录、数据库和日志目录是否有写权限。
- 首次模型加载为何较慢。
- CPU 推理速度、上下文长度和内存之间的关系。
- 修改嵌入模型后为什么必须重建向量索引。

当前仓库中有本地部署说明和安装脚本，但并未看到完整 Docker 化作为主运行方式。因此面试时可以说掌握或了解 Docker 的前提是自己确实能够编写 Dockerfile、Compose 配置并完成运行验证。

## 工程质量、测试与异常处理

项目采用 pytest 覆盖路由、工具、文档解析、检索、数据库迁移、SSE、记忆、知识图谱、批处理、监控和前端契约。测试的意义不仅是证明代码能运行，还用于锁定接口协议和回归行为。

主要工程措施包括：

- FastAPI 全局异常处理，避免把内部堆栈直接返回给用户。
- 工具执行超时与异常捕获。
- DAG 节点重试和备用工具。
- LLM 不可用或超时时回退规则摘要。
- Embedding 依赖缺失时返回友好运维错误。
- 文件入库状态记录为 parsing、done 或 error。
- 数据库迁移幂等执行。
- 日志同时输出到控制台和文件。

面试时应能区分：

| 类型 | 示例 | 处理方式 |
| --- | --- | --- |
| 用户输入错误 | 文件格式不支持、参数缺失 | 4xx 与明确错误信息 |
| 依赖不可用 | 模型未安装、文件缺失 | 503 或可操作的运维提示 |
| 暂时性故障 | 超时、数据库锁 | 重试、回退或稍后再试 |
| 程序缺陷 | 未处理异常 | 记录堆栈，外部返回通用 500 |

**面试提示**：如果你没有亲自写完所有测试，不要说“全部由我独立编写”。可以说你参与设计、补充和运行测试，并说明自己实际负责过哪些部分。

## 项目的不足与改进方向

能主动指出边界并给出合理改进，通常比宣称项目已经完美更有说服力。

| 当前不足 | 影响 | 改进方案 |
| --- | --- | --- |
| 规划主要由规则模板生成 | 新任务适应性不足 | 使用 LLM 结构化输出生成计划，并做白名单和 DAG 校验 |
| 主 RAG 只有向量召回 | 标准编号和专业词精确匹配可能不足 | 融合 ChromaDB 与 FTS5，使用 RRF 或加权排序 |
| 没有 Reranker | Top-K 中可能存在语义相近但无关内容 | 引入 Cross-Encoder 重排 |
| 缺少系统化 RAG 评测 | 优化缺乏量化依据 | 建立问题集，测 Recall@K、MRR、忠实度和响应时间 |
| SQLite 适合单机但并发有限 | 多用户服务扩展受限 | 迁移 PostgreSQL，增加连接池、权限和事务设计 |
| 原生 JS 状态管理逐渐复杂 | 前端维护成本增加 | 使用 React 组件化重构 |
| SQLite、ChromaDB、FTS5 双/三写 | 可能出现数据不一致 | 幂等入库、补偿事务、一致性扫描和重建索引 |
| 模拟文本流 | 首字延迟没有真正降低 | 使用模型原生 streaming 接口 |
| 本地模型推理串行 | 并发吞吐受限 | 独立推理服务、队列、批处理或 vLLM |
| 缺少完整容器化验证 | 部署环境一致性不足 | Dockerfile + Compose + 健康检查 + 持久卷 |

推荐优先升级顺序：

1. 建立 RAG 评测集。
2. 增加 FTS5 与向量融合检索。
3. 增加 Reranker 和引用对齐。
4. 用 React 重构核心页面。
5. 使用 Docker Compose 完成可复现部署。
6. 再考虑 PostgreSQL 和动态 Tool Calling。

## 高频面试问题与回答要点

### 为什么使用 LangGraph，而不是普通函数调用

任务包含意图识别、规划、多工具执行、循环、条件分支、状态累计和最终合成。LangGraph 可以把节点和路由显式表示，统一维护 AgentState，也方便展示执行进度和扩展恢复机制。简单任务可以用普通函数，但复杂工作流用图更清晰。

### 为什么同时使用 SQLite 和 ChromaDB

SQLite 负责结构化业务数据、事务、关联和普通查询；ChromaDB 负责向量相似度检索。两者职责不同，不能互相完全替代。

### 为什么选择 BGE

项目以中文标准文档为主，BGE 中文模型对中文语义检索支持较好，并且可以本地运行，符合断网部署需求。向量经过归一化，便于使用相似度度量。

### Top-K 如何选择

Top-K 太小可能漏掉依据，太大会引入噪声并占用上下文。当前使用配置默认值，合理做法是基于评测集比较不同 K 的召回率、回答质量和延迟，再确定默认值或按问题动态调整。

### 如何降低大模型幻觉

先检索原文，提示模型只能依据工具结果回答；展示来源；检索不到时明确说明；工具失败时不编造；后续还可增加重排、引用对齐和忠实度检测。

### 工具执行失败怎么办

线性工具执行有超时和异常捕获；DAG 节点支持重试和备用工具；最终合成会标明失败步骤；LLM 不可用时回退到规则摘要。

### SSE 与 WebSocket 有什么区别

SSE 是服务器到浏览器的单向事件流，基于 HTTP，适合 Agent 进度和文本推送；WebSocket 是全双工连接，适合双方持续高频交互。项目只需要请求后持续接收结果，因此 SSE 更简单。

### 项目是真正的自主 Tool Calling 吗

当前是规则规划驱动的工具调用：意图和复杂度决定预定义计划，LangGraph 负责执行和状态管理。这样更稳定、可解释。下一步可以让 LLM 输出结构化计划，但仍需工具白名单、参数校验、最大步数和人工确认。

### 项目中最难的问题是什么

可从三个方向选择自己真正做过的内容回答：

1. 把文档解析、向量库和业务数据库保持一致。
2. 让多步骤 Agent 的状态、超时、失败和前端进度正确对应。
3. 在本地模型速度和资源受限的情况下设计降级路径。

回答必须使用 STAR：场景、任务、行动、结果，并准备具体代码或故障案例。

## 学习与验收清单

按下面顺序阅读代码，不要从所有工具类开始逐个看。

### 第一阶段：Agent 主链路

1. `backend/routes/agent.py`
2. `agent/orchestrator.py`
3. `agent/intent.py`
4. `agent/planner.py`
5. `agent/executor.py`
6. `agent/tools/base.py`
7. `agent/tools/registry.py`
8. `backend/utils/sse.py`
9. `static/js/chat.js`

验收：不看代码画出对话流程图，并回答节点、状态、路由、工具、超时和 SSE 的作用。

### 第二阶段：文档与 RAG

1. `backend/services/parser.py`
2. `document/pipeline.py`
3. `document/cleaner.py`
4. `document/chunker.py`
5. `backend/services/embedder.py`
6. `backend/database/operations.py`
7. `backend/services/fts_search.py`
8. `agent/tools/knowledge/search.py`
9. `agent/tools/knowledge/qa.py`

验收：选择一个测试文档，说明它最终被切成哪些块、向量和原文分别存在哪里，以及一个问题如何找到相关块。

### 第三阶段：工程与部署

1. `backend/main.py`
2. `backend/config/settings.py`
3. `backend/database/connection.py`
4. `backend/database/schema.py`
5. `backend/services/llm_client.py`
6. `tests/`
7. `docs/LOCAL_DEPLOYMENT.md`

验收：从空白终端启动项目，定位日志，解释数据库初始化、模型加载、常见错误和关闭流程。

### 必须亲手完成的练习

- 新增一个最简单的 Agent 工具并注册。
- 新增一个 FastAPI 接口及对应测试。
- 给一个工具制造超时，观察 SSE 错误和回退。
- 用同一问题比较 Top-K 为 3、5、10 的检索结果。
- 使用 FTS5 和 ChromaDB 分别搜索同一关键词，比较差异。
- 从零启动项目并录制三分钟演示。
- 准备一个真实 Bug：现象、定位过程、根因、修复和验证。

完成以上内容后，才算真正具备面试中讲解该项目的能力。